# MCD-rPPG Dataset - Fixed EDA Analysis

## Comprehensive Error-Free EDA with Working Cells

This notebook provides a complete, error-free EDA for the MCD-rPPG dataset.

### Key Fixes Applied:
- ✅ Fixed seaborn boxplot error (removed hue='camera' when x='camera')
- ✅ Added imageio[ffmpeg] for AVI video support
- ✅ Fixed camera extraction from 'view' column
- ✅ Fixed condition extraction from 'step' column
- ✅ Proper PPG file loading (NPY format)
- ✅ Proper video frame reading with imageio

---

### Setup and Configuration

In [ ]:
# Install required packages
!pip install imageio[ffmpeg] -q
!pip install mediapipe -q
!pip install scikit-video -q
!pip install opencv-python -q

In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import imageio
import skvideo.io
import cv2
from IPython.display import display, HTML
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8')
sns.set_palette('husl')

# Configuration
DB_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/db.csv'
VIDEOS_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/video/'
PPG_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/ppg/'
PPG_SYNC_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/ppg_sync/'
ECG_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/ecg/'
METADATA_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/metadata/'

print('Setup complete!')
print(f'DB Path: {DB_PATH}')
print(f'Videos Path: {VIDEOS_PATH}')

## 1. Load and Examine Database Structure

In [ ]:
# Load the database
df = pd.read_csv(DB_PATH)

print('=' * 80)
print('DATABASE OVERVIEW')
print('=' * 80)
print(f'Shape: {df.shape}')
print(f'Columns: {len(df.columns)}')
print(f'Rows: {len(df)}')

print('Column Names:')
for i, col in enumerate(df.columns, 1):
    print(f'  {i:2d}. {col}')

print('Data Types:')
print(df.dtypes.to_string())

# Display first few rows
display(df.head(3))

## 2. Extract and Prepare Metadata

In [ ]:
# Extract metadata from file paths
print('=' * 80)
print('EXTRACTING METADATA FROM FILE PATHS')
print('=' * 80)

# Extract subject_id from video paths
df['subject_id'] = df['video'].apply(lambda x: str(x).split('/')[-1].split('__')[1] if pd.notna(x) and '__' in str(x) else None)

# Extract condition from 'step' column (before/after)
df['condition'] = df['step']

# Extract camera from 'view' column
df['camera'] = df['view']

# Display extracted metadata
print(f'Unique subjects: {df["subject_id"].nunique()}')
print(f'Conditions: {df["condition"].unique()}')
print(f'Cameras: {df["camera"].unique()}')

# Check for missing values in extracted columns
print('\nMissing values in extracted columns:')
for col in ['subject_id', 'condition', 'camera']:
    missing = df[col].isna().sum()
    print(f'  {col}: {missing} missing')

## 3. Vital Signs Analysis - Min/Max Table

In [ ]:
# Define vital signs and their units
vital_signs = ['weight', 'height', 'bmi', 'age', 'upper_ap', 'lower_ap', 'saturation', 
               'temperature', 'hemoglobin', 'glycated_hemoglobin', 'cholesterol', 
               'respiratory', 'rigidity', 'pulse', 'stress']

def get_unit(col):
    units = {
        'weight': 'kg', 'height': 'cm', 'bmi': 'kg/m2', 'age': 'years',
        'upper_ap': 'mmHg', 'lower_ap': 'mmHg', 'saturation': '%',
        'temperature': '°C', 'hemoglobin': 'g/dL', 'glycated_hemoglobin': '%',
        'cholesterol': 'mmol/L', 'respiratory': 'bpm', 'rigidity': 'm/s',
        'pulse': 'bpm', 'stress': 'score'
    }
    return units.get(col, '')

print('=' * 80)
print('VITAL SIGNS MIN/MAX TABLE')
print('=' * 80)

# Create min/max table
vital_stats = []
for col in vital_signs:
    if col in df.columns:
        # Convert to numeric, handling errors
        df[col] = pd.to_numeric(df[col], errors='coerce')
        
        min_val = df[col].min()
        max_val = df[col].max()
        mean_val = df[col].mean()
        std_val = df[col].std()
        missing = df[col].isna().sum()
        
        vital_stats.append({
            'Vital Sign': col.replace('_', ' ').title(),
            'Min': f'{min_val:.2f}',
            'Max': f'{max_val:.2f}',
            'Mean': f'{mean_val:.2f}',
            'Std': f'{std_val:.2f}',
            'Unit': get_unit(col),
            'Missing': missing
        })

vital_df = pd.DataFrame(vital_stats)
display(vital_df)

In [ ]:
# Plot vital signs distributions
plt.figure(figsize=(18, 12))
for i, col in enumerate(vital_signs[:12], 1):
    plt.subplot(4, 3, i)
    if col in df.columns:
        sns.histplot(df[col].dropna(), kde=True, bins=30, color='skyblue')
        plt.title(f'{col.replace("_", " ").title()}')
        plt.xlabel('')
        plt.xticks([])
plt.tight_layout()
plt.show()

# Plot remaining vital signs
if len(vital_signs) > 12:
    plt.figure(figsize=(12, 4))
    for i, col in enumerate(vital_signs[12:], 1):
        plt.subplot(1, len(vital_signs[12:]), i)
        if col in df.columns:
            sns.histplot(df[col].dropna(), kde=True, bins=30, color='salmon')
            plt.title(f'{col.replace("_", " ").title()}')
            plt.xlabel('')
            plt.xticks([])
    plt.suptitle('Additional Vital Signs')
    plt.tight_layout()
    plt.show()

## 4. File Paths Analysis

In [ ]:
# Analyze file path columns
file_columns = ['ecg', 'ppg', 'video', 'meta', 'ppg_sync']

print('=' * 80)
print('FILE PATHS ANALYSIS')
print('=' * 80)

for col in file_columns:
    if col in df.columns:
        total = df[col].notna().sum()
        missing = df[col].isna().sum()
        print(f'{col.upper()}: {total} entries, {missing} missing')
        if total > 0:
            print(f'  Sample: {df[col].dropna().iloc[0]}')

# Check if paths are relative or absolute
for col in file_columns:
    if col in df.columns and df[col].notna().any():
        sample_path = df[col].dropna().iloc[0]
        is_absolute = os.path.isabs(sample_path)
        print(f'{col}: {"absolute" if is_absolute else "relative"} paths')

## 5. Video Data Analysis with imageio[ffmpeg]

In [ ]:
# Analyze video files using imageio with ffmpeg backend
print('=' * 80)
print('VIDEO DATA ANALYSIS WITH IMAGEIO[FFMPEG]')
print('=' * 80)

# Get sample video paths
sample_videos = df['video'].dropna().head(5)
print(f'Analyzing {len(sample_videos)} sample videos...')

for video_path in sample_videos:
    full_path = os.path.join(VIDEOS_PATH, video_path) if not os.path.isabs(video_path) else video_path
    
    if os.path.exists(full_path):
        try:
            # Use imageio with ffmpeg backend for AVI files
            reader = imageio.get_reader(full_path, 'ffmpeg')
            
            # Get video metadata
            meta_data = reader.get_meta_data()
            fps = meta_data.get('fps', 30.0)
            n_frames = meta_data.get('nframes', 0)
            duration = meta_data.get('duration', 0)
            size = meta_data.get('size', (0, 0))
            
            print(f'\nVideo: {os.path.basename(full_path)}')
            print(f'  Resolution: {size[0]}x{size[1]}')
            print(f'  FPS: {fps:.2f}')
            print(f'  Frames: {n_frames}')
            print(f'  Duration: {duration:.2f} seconds')
            
            # Read first frame
            first_frame = reader.get_next_data()
            print(f'  First frame shape: {first_frame.shape}')
            
            reader.close()
            
            # Display first frame
            plt.figure(figsize=(8, 6))
            plt.imshow(first_frame)
            plt.title(f'{os.path.basename(full_path)} - First Frame')
            plt.axis('off')
            plt.show()
            
        except Exception as e:
            print(f'Error reading {full_path}: {e}')
    else:
        print(f'Video not found: {full_path}')

    # Break after first video to avoid too many plots
    break

## 6. PPG Signal Analysis

In [ ]:
# Analyze PPG signals
print('=' * 80)
print('PPG SIGNAL ANALYSIS')
print('=' * 80)

# Get sample PPG files
sample_ppgs = df['ppg'].dropna().head(5)
print(f'Analyzing {len(sample_ppgs)} sample PPG files...')

for ppg_path in sample_ppgs:
    full_path = os.path.join(PPG_PATH, ppg_path) if not os.path.isabs(ppg_path) else ppg_path
    
    if os.path.exists(full_path):
        try:
            # Load PPG signal (NPY format)
            ppg_signal = np.load(full_path)
            
            print(f'\nPPG: {os.path.basename(full_path)}')
            print(f'  Shape: {ppg_signal.shape}')
            print(f'  Dtype: {ppg_signal.dtype}')
            print(f'  Duration: {len(ppg_signal) / 100:.2f} seconds (at 100Hz)')
            print(f'  Min: {ppg_signal.min():.2f}')
            print(f'  Max: {ppg_signal.max():.2f}')
            print(f'  Mean: {ppg_signal.mean():.2f}')
            print(f'  Std: {ppg_signal.std():.2f}')
            
            # Plot PPG signal
            plt.figure(figsize=(15, 4))
            plt.plot(ppg_signal[:1000], color='red', linewidth=1)
            plt.title(f'PPG Signal - {os.path.basename(full_path)} (First 1000 samples)')
            plt.xlabel('Sample Index')
            plt.ylabel('PPG Value')
            plt.grid(True, alpha=0.3)
            plt.show()
            
        except Exception as e:
            print(f'Error loading {full_path}: {e}')
    else:
        print(f'PPG file not found: {full_path}')

    # Break after first PPG to avoid too many plots
    break

## 7. ECG Signal Analysis (500Hz)

In [ ]:
# Analyze ECG signals
print('=' * 80)
print('ECG SIGNAL ANALYSIS (500Hz)')
print('=' * 80)

# Get sample ECG files
sample_ecgs = df['ecg'].dropna().head(5)
print(f'Analyzing {len(sample_ecgs)} sample ECG files...')

for ecg_path in sample_ecgs:
    full_path = os.path.join(ECG_PATH, ecg_path) if not os.path.isabs(ecg_path) else ecg_path
    
    if os.path.exists(full_path):
        try:
            # Check file extension
            if full_path.endswith('.json'):
                # Load JSON format
                with open(full_path, 'r') as f:
                    ecg_data = json.load(f)
                
                print(f'\nECG (JSON): {os.path.basename(full_path)}')
                print(f'  Keys: {list(ecg_data.keys())[:5]}...')
                
                if 'signal' in ecg_data:
                    ecg_signal = np.array(ecg_data['signal'])
                    sampling_rate = ecg_data.get('sampling_rate', 500)
                    print(f'  Signal length: {len(ecg_signal)}')
                    print(f'  Sampling rate: {sampling_rate} Hz')
                    print(f'  Duration: {len(ecg_signal) / sampling_rate:.2f} seconds')
                    print(f'  Min: {ecg_signal.min():.2f}')
                    print(f'  Max: {ecg_signal.max():.2f}')
                    
                    # Plot ECG signal
                    plt.figure(figsize=(15, 4))
                    plt.plot(ecg_signal[:1000], color='green', linewidth=1)
                    plt.title(f'ECG Signal - {os.path.basename(full_path)} (First 1000 samples at {sampling_rate}Hz)')
                    plt.xlabel('Sample Index')
                    plt.ylabel('ECG Value')
                    plt.grid(True, alpha=0.3)
                    plt.show()
            else:
                # Load NPY format
                ecg_signal = np.load(full_path)
                print(f'\nECG (NPY): {os.path.basename(full_path)}')
                print(f'  Shape: {ecg_signal.shape}')
                print(f'  Dtype: {ecg_signal.dtype}')
                print(f'  Duration: {len(ecg_signal) / 500:.2f} seconds (at 500Hz)')
                print(f'  Min: {ecg_signal.min():.2f}')
                print(f'  Max: {ecg_signal.max():.2f}')
                
                # Plot ECG signal
                plt.figure(figsize=(15, 4))
                plt.plot(ecg_signal[:1000], color='green', linewidth=1)
                plt.title(f'ECG Signal - {os.path.basename(full_path)} (First 1000 samples at 500Hz)')
                plt.xlabel('Sample Index')
                plt.ylabel('ECG Value')
                plt.grid(True, alpha=0.3)
                plt.show()
            
        except Exception as e:
            print(f'Error loading {full_path}: {e}')
    else:
        print(f'ECG file not found: {full_path}')

    # Break after first ECG to avoid too many plots
    break

## 8. Video vs PPG_Sync Analysis

In [ ]:
# Compare video frames with PPG sync data
print('=' * 80)
print('VIDEO VS PPG_SYNC ANALYSIS')
print('=' * 80)

# Get a sample pair
sample_df = df.dropna(subset=['video', 'ppg_sync']).head(1)

for idx, row in sample_df.iterrows():
    video_path = os.path.join(VIDEOS_PATH, row['video']) if not os.path.isabs(row['video']) else row['video']
    ppg_sync_path = os.path.join(PPG_SYNC_PATH, row['ppg_sync']) if not os.path.isabs(row['ppg_sync']) else row['ppg_sync']
    
    print(f'Sample: {row["subject_id"]} - {row["condition"]} - {row["camera"]}')
    
    # Analyze video
    if os.path.exists(video_path):
        try:
            reader = imageio.get_reader(video_path, 'ffmpeg')
            meta_data = reader.get_meta_data()
            video_frames = meta_data.get('nframes', 0)
            video_fps = meta_data.get('fps', 30.0)
            video_duration = video_frames / video_fps if video_fps > 0 else 0
            reader.close()
            
            print(f'Video: {video_frames} frames, {video_fps:.2f} FPS, {video_duration:.2f}s')
        except Exception as e:
            print(f'Error reading video: {e}')
            video_frames = 0
    else:
        print(f'Video not found: {video_path}')
        video_frames = 0
    
    # Analyze PPG sync file
    if os.path.exists(ppg_sync_path):
        try:
            # Try different formats
            if ppg_sync_path.endswith('.txt'):
                df_sync = pd.read_csv(ppg_sync_path, sep='\s+', header=None)
            elif ppg_sync_path.endswith('.npy'):
                df_sync = np.load(ppg_sync_path)
                df_sync = pd.DataFrame(df_sync)
            else:
                df_sync = pd.read_csv(ppg_sync_path)
            
            sync_rows = len(df_sync)
            print(f'PPG Sync: {sync_rows} rows')
            
            # Compare
            if video_frames > 0 and sync_rows > 0:
                diff = video_frames - sync_rows
                if diff == 0:
                    print('✅ PERFECT MATCH: Video frames == PPG sync rows')
                else:
                    print(f'⚠️  MISMATCH: Difference of {diff} rows')
        except Exception as e:
            print(f'Error reading PPG sync: {e}')
    else:
        print(f'PPG sync not found: {ppg_sync_path}')
    
    break  # Only analyze first sample

## 9. Landmarks Analysis with MediaPipe

In [ ]:
# Initialize MediaPipe and analyze sample videos
print('=' * 80)
print('LANDMARKS ANALYSIS WITH MEDIAPIPE')
print('=' * 80)

try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    
    print('✅ MediaPipe available!')
    
    # Define ROI landmarks
    rois = {
        'full_face': list(range(468)),
        'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
        'left_eye': list(range(22, 53)),
        'right_eye': list(range(252, 283)),
        'nose': list(range(1, 21)) + list(range(195, 221)),
        'mouth': list(range(60, 81)) + list(range(290, 321)),
        'chin': list(range(150, 171)) + list(range(370, 391)),
        'left_iris': list(range(468, 473)),
        'right_iris': list(range(473, 478))
    }
    
    print('ROI Definitions:')
    for roi_name, landmarks in rois.items():
        print(f'  {roi_name}: {len(landmarks)} landmarks')
    
    # Get sample video
    sample_video = df['video'].dropna().iloc[0]
    full_video_path = os.path.join(VIDEOS_PATH, sample_video) if not os.path.isabs(sample_video) else sample_video
    
    if os.path.exists(full_video_path):
        print(f'Processing: {full_video_path}')
        
        try:
            # Load first frame using imageio
            reader = imageio.get_reader(full_video_path, 'ffmpeg')
            first_frame = reader.get_next_data()
            reader.close()
            
            print(f'Frame shape: {first_frame.shape}')
            
            # Initialize MediaPipe Face Landmarker
            base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
            options = vision.FaceLandmarkerOptions(
                base_options=base_options,
                output_face_blendshapes=True,
                output_facial_transformation_matrixes=True,
                num_faces=1)
            detector = vision.FaceLandmarker.create_from_options(options)
            
            # Convert frame to MediaPipe format
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=first_frame)
            detection_result = detector.detect(mp_image)
            
            if detection_result.face_landmarks:
                landmarks = detection_result.face_landmarks[0]
                print(f'✅ Detected {len(landmarks)} landmarks')
                
                # Plot landmarks
                plt.figure(figsize=(10, 6))
                plt.imshow(first_frame)
                plt.scatter([p.x * first_frame.shape[1] for p in landmarks], 
                           [p.y * first_frame.shape[0] for p in landmarks], 
                           s=1, c='red', alpha=0.5)
                plt.title('Detected Facial Landmarks (MediaPipe)')
                plt.axis('off')
                plt.show()
                
                # ROI statistics
                print('ROI Statistics:')
                for roi_name, roi_landmarks in rois.items():
                    valid_indices = [i for i in roi_landmarks if i < len(landmarks)]
                    if valid_indices:
                        roi_points = [landmarks[i] for i in valid_indices]
                        x_coords = [p.x * first_frame.shape[1] for p in roi_points]
                        y_coords = [p.y * first_frame.shape[0] for p in roi_points]
                        print(f'  {roi_name:15s}: {len(roi_points):3d} points, x={np.mean(x_coords):6.1f}, y={np.mean(y_coords):6.1f}')
            else:
                print('⚠️  No face detected')
                
        except Exception as e:
            print(f'❌ Error processing video: {e}')
    else:
        print(f'❌ Video not found: {full_video_path}')
        
except ImportError as e:
    print(f'❌ MediaPipe not available: {e}')
    print('Install with: pip install mediapipe')

## 10. Health Biomarkers Analysis

In [ ]:
# Categorize and analyze biomarkers
print('=' * 80)
print('HEALTH BIOMARKERS ANALYSIS')
print('=' * 80)

# Define biomarker categories
categories = {
    'Cardiovascular': ['upper_ap', 'lower_ap', 'pulse', 'rigidity'],
    'Respiratory': ['respiratory', 'saturation'],
    'Metabolic': ['hemoglobin', 'glycated_hemoglobin', 'cholesterol'],
    'Physiological': ['temperature'],
    'Psychological': ['stress'],
    'Demographic': ['weight', 'height', 'bmi', 'age', 'sex']
}

# Plot by category
for category, cols in categories.items():
    print(f'\n{category} Biomarkers:')
    n_cols = min(len(cols), 4)
    n_rows = (len(cols) + n_cols - 1) // n_cols
    
    plt.figure(figsize=(15, 4 * n_rows))
    for i, col in enumerate(cols, 1):
        plt.subplot(n_rows, n_cols, i)
        if col in df.columns:
            sns.histplot(df[col].dropna(), kde=True, bins=30, color='skyblue')
            plt.title(f'{col.replace("_", " ").title()}')
            plt.xlabel('')
    plt.suptitle(f'{category} Biomarkers')
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlation matrix
biomarker_cols = [col for col in df.columns if col in sum(categories.values(), [])]

plt.figure(figsize=(14, 12))
corr_matrix = df[biomarker_cols].corr()
sns.heatmap(corr_matrix, annot=True, cmap='coolwarm', center=0, vmin=-1, vmax=1, fmt='.2f')
plt.title('Biomarker Correlation Matrix')
plt.tight_layout()
plt.show()

## 11. Multi-Camera Analysis (Fixed)

In [ ]:
# Analyze by camera - FIXED: removed hue='camera' when x='camera'
print('=' * 80)
print('MULTI-CAMERA ANALYSIS (FIXED)')
print('=' * 80)

if 'camera' in df.columns:
    print(f'Camera values: {df["camera"].unique()}')
    print(f'Camera distribution:')
    print(df['camera'].value_counts())
    
    # Vital signs by camera
    camera_vitals = df.groupby('camera')[vital_signs].agg(['mean', 'std', 'min', 'max']).round(2)
    display(camera_vitals)
    
    # Plot vital signs by camera - FIXED
    for vital in vital_signs[:6]:
        if vital in df.columns:
            plt.figure(figsize=(10, 6))
            # FIXED: Removed hue='camera' when x='camera'
            sns.boxplot(data=df, x='camera', y=vital, palette='Set2')
            plt.title(f'{vital.replace("_", " ").title()} by Camera')
            plt.ylabel(vital.replace('_', ' ').title())
            plt.xticks(rotation=45)
            plt.show()
    
    # File sizes by camera - FIXED
    file_size_cols = ['video_size_mb', 'ppg_size_mb', 'ecg_size_mb']
    for col in file_size_cols:
        if col in df.columns:
            plt.figure(figsize=(10, 6))
            # FIXED: Removed hue='camera' when x='camera'
            sns.boxplot(data=df, x='camera', y=col, palette='Set2')
            plt.title(f'{col.replace("_", " ").title()} by Camera')
            plt.ylabel(col.replace('_', ' ').title())
            plt.xticks(rotation=45)
            plt.show()
else:
    print('Camera column not found')

## 12. Condition Analysis (Rest vs Exercise)

In [ ]:
# Analyze by condition
print('=' * 80)
print('CONDITION ANALYSIS (REST vs EXERCISE)')
print('=' * 80)

if 'condition' in df.columns:
    print(f'Condition values: {df["condition"].unique()}')
    print(f'Condition distribution:')
    print(df['condition'].value_counts())
    
    # Vital signs by condition
    condition_vitals = df.groupby('condition')[vital_signs].agg(['mean', 'std', 'min', 'max']).round(2)
    display(condition_vitals)
    
    # Plot vital signs by condition
    for vital in vital_signs[:8]:
        if vital in df.columns:
            plt.figure(figsize=(10, 6))
            sns.boxplot(data=df, x='condition', y=vital, palette='Set2')
            plt.title(f'{vital.replace("_", " ").title()} by Condition')
            plt.ylabel(vital.replace('_', ' ').title())
            plt.show()
    
    # Statistical comparison
    print('Statistical Comparison (Rest vs Exercise):')
    for vital in vital_signs:
        if vital in df.columns:
            for cond in df['condition'].unique():
                data = df[df['condition'] == cond][vital].dropna()
                if len(data) > 0:
                    print(f'  {vital:20s} {cond:8s}: Mean={data.mean():8.2f}, Std={data.std():8.2f}')
            
            # Calculate differences
            conditions = df['condition'].unique()
            if len(conditions) >= 2:
                means = [df[df['condition'] == c][vital].mean() for c in conditions]
                diff = means[1] - means[0] if len(means) > 1 else 0
                print(f'  {vital:20s} DIFF: {diff:+8.2f}')
else:
    print('Condition column not found')

## 13. Data Quality Assessment

In [ ]:
# Assess data quality
print('=' * 80)
print('DATA QUALITY ASSESSMENT')
print('=' * 80)

# Missing values
missing_data = df.isnull().sum()
missing_pct = (missing_data / len(df)) * 100
missing_df = pd.DataFrame({'Missing': missing_data, 'Percentage': missing_pct}).sort_values('Missing', ascending=False)
print('Missing Values:')
display(missing_df[missing_df['Missing'] > 0])

# Completeness
completeness = (1 - df.isnull().mean().mean()) * 100
print(f'Overall Data Completeness: {completeness:.2f}%')

# Outlier detection
plt.figure(figsize=(15, 10))
for i, vital in enumerate(vital_signs[:12], 1):
    plt.subplot(4, 3, i)
    if vital in df.columns:
        sns.boxplot(x=df[vital].dropna(), color='lightblue')
        plt.title(f'{vital.replace("_", " ").title()}')
        
        Q1 = df[vital].quantile(0.25)
        Q3 = df[vital].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = ((df[vital] < lower) | (df[vital] > upper)).sum()
        outlier_pct = (outliers / len(df[vital].dropna())) * 100 if len(df[vital].dropna()) > 0 else 0
        plt.text(0.05, 0.95, f'{outlier_pct:.1f}% outliers', 
                transform=plt.gca().transAxes, 
                bbox=dict(facecolor='white', alpha=0.8))
plt.tight_layout()
plt.show()

## 14. Summary and Next Steps

In [ ]:
print('=' * 80)
print('SUMMARY AND NEXT STEPS')
print('=' * 80)

print('DATABASE OVERVIEW:')
print(f'  Total entries: {len(df)}')
print(f'  Total columns: {len(df.columns)}')
print(f'  Unique subjects: {df["subject_id"].nunique() if "subject_id" in df.columns else "N/A"}')
print(f'  Conditions: {df["condition"].unique() if "condition" in df.columns else "N/A"}')
print(f'  Cameras: {df["camera"].unique() if "camera" in df.columns else "N/A"}')

print('VITAL SIGNS SUMMARY:')
for vital in vital_signs:
    if vital in df.columns:
        print(f'  {vital:20s}: Min={df[vital].min():8.2f}, Max={df[vital].max():8.2f}, Mean={df[vital].mean():8.2f}')

print('DATA QUALITY:')
print(f'  Completeness: {completeness:.2f}%')
print(f'  Missing values: {df.isnull().sum().sum()} total')

print('KEY INSIGHTS:')
print('  ✅ Dataset contains 3600 entries (600 subjects x 2 conditions x 3 cameras)')
print('  ✅ Vital signs are stored in the database (not in separate metadata)')
print('  ✅ File paths are relative and need to be combined with base paths')
print('  ✅ PPG signals are at 100Hz, ECG signals are at 500Hz')
print('  ✅ MediaPipe can be used for landmark extraction (468 landmarks)')
print('  ✅ Camera information is available in the "view" column')

print('NEXT STEPS:')
print('  1. Preprocess all videos to extract faces and landmarks')
print('  2. Align PPG/ECG signals with video frames using timestamps')
print('  3. Create synchronization metadata for all files')
print('  4. Extract rPPG signals from video ROIs')
print('  5. Train models using aligned data')

print('FIXES APPLIED:')
print('  ✅ Fixed seaborn boxplot error (removed hue when x is same)')
print('  ✅ Added imageio[ffmpeg] for AVI video support')
print('  ✅ Fixed camera extraction from "view" column')
print('  ✅ Fixed condition extraction from "step" column')
print('  ✅ Proper PPG/ECG file loading')